# Logistic Regression (From Scratch)

Eksperimen model `LogisticRegression` *from scratch* (optimasi SGA & Adam serta regularisasi L1, L2, dan Elastic Net), perbandingan dengan `scikit-learn`, dan pembuatan submisi.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.evaluation import (
    cross_validate_configs,
    generate_submission,
    save_and_verify_model,
    split_train_validation,
    train_and_evaluate,
)
from src.models import (
    DEFAULT_LOGREG_CONFIGS,
    LogisticRegression,
    get_logreg_preprocessing_config,
)
from src.preprocess.preprocess import run_preprocessing

RANDOM_STATE = 824

In [ ]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

logreg_prep = run_preprocessing(
    train_df,
    test_df,
    config=get_logreg_preprocessing_config(random_state=RANDOM_STATE, verbose=True),
)
logreg_split = split_train_validation(
    logreg_prep.X_train,
    logreg_prep.y_train,
    is_orig=logreg_prep.is_orig,
    test_size=0.15,
    random_state=RANDOM_STATE,
    verbose=True,
)

## 5-Fold Stratified Cross-Validation (SGA & Adam, None / L1 / L2 / Elastic Net)

In [ ]:
best_logreg_config, logreg_cv_df = cross_validate_configs(
    LogisticRegression,
    DEFAULT_LOGREG_CONFIGS,
    logreg_split.X_tr,
    logreg_split.y_tr,
    k_folds=5,
    random_state=RANDOM_STATE,
    model_name="Logistic Regression",
    verbose=True,
)
display(logreg_cv_df)

logreg_model_cfg = {k: v for k, v in best_logreg_config.items() if k != "prob_thres"}
logreg_result = train_and_evaluate(
    LogisticRegression(**logreg_model_cfg),
    logreg_split,
    model_name="Logistic Regression",
    config=best_logreg_config,
    X_test=logreg_prep.X_test,
    fit_orig_only=True,
    plot_cm=True,
)
best_model = logreg_result["model"]

## Perbandingan dengan Scikit-Learn

In [ ]:
X_tr_orig = logreg_split.X_tr[logreg_split.orig_tr]
y_tr_orig = logreg_split.y_tr[logreg_split.orig_tr]
sk_logreg = SklearnLogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
sk_logreg.fit(X_tr_orig, y_tr_orig)

X_val_orig = logreg_split.X_val[logreg_split.orig_val]
y_val_orig = logreg_split.y_val[logreg_split.orig_val]
scratch_preds = best_model.predict(X_val_orig)
sk_preds = sk_logreg.predict(X_val_orig)

def eval_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

comp_df = pd.DataFrame([
    eval_row("LogisticRegression (Scratch)", y_val_orig, scratch_preds),
    eval_row("LogisticRegression (Scikit-Learn)", y_val_orig, sk_preds),
])
display(comp_df)

agreement = (scratch_preds == sk_preds).mean() * 100.0
w_s, w_k = best_model.coef_, sk_logreg.coef_.ravel()
cos_sim = float(np.dot(w_s, w_k) / (np.linalg.norm(w_s) * np.linalg.norm(w_k)))
print(f"Prediction Agreement: {agreement:.2f}% | Weight Cosine Similarity: {cos_sim:.5f}")

## Save, Load, dan Submission

In [ ]:
loaded_model = save_and_verify_model(best_model, "output/logreg.pkl", verbose=True)
assert np.allclose(best_model.b_, loaded_model.b_)
submission = generate_submission(
    loaded_model,
    logreg_prep.X_test,
    test_df["id"],
    sample_submission,
    "output/submission_logreg.csv",
    random_state=RANDOM_STATE,
    verbose=True,
)
display(submission.head())